In [1]:
import os
import json
import yaml
from src.utils.file_utils import find_config_with_conditions

In [2]:
from torch.optim.lr_scheduler import ReduceLROnPlateau, LRScheduler
print(issubclass(ReduceLROnPlateau, LRScheduler))  # 输出 True 表示是子类

True


In [3]:
def match_conditions(config, conditions):
    """
    递归匹配 config 是否满足 conditions 条件
    """
    for k, v in conditions.items():
        if k not in config:
            return False
        if isinstance(v, dict):
            if not isinstance(config[k], dict):
                return False
            if not match_conditions(config[k], v):
                return False
        else:
            if config[k] != v:
                return False
    return True

def find_config_with_conditions(conditions, root_dir):
    """
    根据给定的条件递归筛选 config.yaml 文件，并返回所有符合条件的文件夹绝对路径
    """
    matching_dirs = []

    for subdir, _, files in os.walk(root_dir):
        if 'config.yaml' in files:
            config_path = os.path.join(subdir, 'config.yaml')
            try:
                with open(config_path, 'r') as f:
                    config = yaml.safe_load(f)

                if match_conditions(config, conditions):
                    matching_dirs.append(os.path.abspath(subdir))
            except Exception as e:
                print(f"[错误] 读取 {config_path} 时失败: {e}")

    return matching_dirs


In [5]:
root_dir = '/root/autodl-tmp/chuandian_eq/checkpoints'
conditions = {
    'model': 'clf_mixer_attnpl_t',
    'mixer_model_config': {
        'd_model': 64,
        # 'attn_layer_idx': [],
        # 'ssm_cfg':{'layer': 'Mamba2'}
    },
    'dataset':'ChinaArray'
}

dirs = find_config_with_conditions(conditions, root_dir)

In [6]:
dirs

['/root/autodl-tmp/chuandian_eq/checkpoints/clf_mixer_attnpl_t_20250807-213332',
 '/root/autodl-tmp/chuandian_eq/checkpoints/clf_mixer_attnpl_t_20250807-215439',
 '/root/autodl-tmp/chuandian_eq/checkpoints/clf_mixer_attnpl_t_20250808-093601',
 '/root/autodl-tmp/chuandian_eq/checkpoints/clf_mixer_attnpl_t_20250808-095331',
 '/root/autodl-tmp/chuandian_eq/checkpoints/clf_mixer_attnpl_t_20250808-101201',
 '/root/autodl-tmp/chuandian_eq/checkpoints/clf_mixer_attnpl_t_20250808-102759',
 '/root/autodl-tmp/chuandian_eq/checkpoints/clf_mixer_attnpl_t_20250808-104823',
 '/root/autodl-tmp/chuandian_eq/checkpoints/clf_mixer_attnpl_t_20250808-110649',
 '/root/autodl-tmp/chuandian_eq/checkpoints/clf_mixer_attnpl_t_20250808-112348',
 '/root/autodl-tmp/chuandian_eq/checkpoints/clf_mixer_attnpl_t_20250808-114141',
 '/root/autodl-tmp/chuandian_eq/checkpoints/clf_mixer_attnpl_t_20250808-115726',
 '/root/autodl-tmp/chuandian_eq/checkpoints/clf_mixer_attnpl_t_20250808-140430',
 '/root/autodl-tmp/chuandian

查找checkpoint

In [8]:
def get_metrics_json_from_file_list(file_list):
    """
    根据给定的文件列表，检查每个文件夹是否包含 metrics.json 文件，并返回其内容
    
    :param file_list: 包含文件夹路径的列表
    :return: 返回一个包含文件夹路径和对应的 metrics.json 数据的字典列表
    """
    result = [] 

    # 遍历 file_list 中的每个文件夹路径
    for folder_path in file_list:
        metrics_path = os.path.join(folder_path, 'metrics.json')  # 拼接 metrics.json 的路径
        if os.path.isfile(metrics_path):  # 如果 metrics.json 文件存在
            try:
                # 读取并解析 metrics.json 文件
                with open(metrics_path, 'r') as metrics_file:
                    metrics_data = json.load(metrics_file)
                
                # 将文件夹路径和对应的 metrics.json 数据添加到结果列表中
                result.append({
                    'folder_path': os.path.abspath(folder_path),
                    'metrics_data': metrics_data
                })
            except Exception as e:
                print(f"读取 {metrics_path} 文件时发生错误: {e}")
        else:
            print(f"文件夹 {folder_path} 中未找到 metrics.json 文件")

    # 返回符合条件的所有文件夹路径和对应的 metrics.json 数据
    return result


json_files = get_metrics_json_from_file_list(dirs)

# 打印结果
if json_files:
    print("找到的文件夹和对应的 metrics.json 内容:")
    for item in json_files:
        print(f"文件夹路径: {item['folder_path']}")
        print(f"metrics.json 内容: {item['metrics_data']}")
else:
    print("没有找到包含 metrics.json 文件的目录")


文件夹 /root/autodl-tmp/chuandian_eq/checkpoints/clf_mixer_attnpl_t_20250815-103333 中未找到 metrics.json 文件
找到的文件夹和对应的 metrics.json 内容:
文件夹路径: /root/autodl-tmp/chuandian_eq/checkpoints/clf_mixer_attnpl_t_20250807-213332
metrics.json 内容: {'precision': 0.6691176470588235, 'recall': 0.9381443298969072, 'f1': 0.7811158798283262, 'auc': 0.692854603626022, 'fpr': 0.7758620689655172, 'tpr': 0.9381443298969072, 'R': 0.16228226093138998, 'conf': 0.0009858269940858865, 'threshold': 0.38063833117485046}
文件夹路径: /root/autodl-tmp/chuandian_eq/checkpoints/clf_mixer_attnpl_t_20250807-215439
metrics.json 内容: {'precision': 0.712, 'recall': 0.9175257731958762, 'f1': 0.8018018018018018, 'auc': 0.7065410593672237, 'fpr': 0.6206896551724138, 'tpr': 0.9175257731958762, 'R': 0.29683611802346244, 'conf': 0.9919078066899294, 'threshold': 0.17336198687553406}
文件夹路径: /root/autodl-tmp/chuandian_eq/checkpoints/clf_mixer_attnpl_t_20250808-093601
metrics.json 内容: {'precision': 0.7543859649122807, 'recall': 0.88659793814432

按日期删除checkpoint

In [45]:
import os
import shutil
from datetime import datetime

# 设置目标目录
checkpoint_dir = "checkpoints"
# 指定保留起点（目标时间），格式必须和文件夹一致
threshold = "20250518-205722"
threshold_dt = datetime.strptime(threshold, "%Y%m%d-%H%M%S")

# 遍历文件夹
for folder in os.listdir(checkpoint_dir):
    folder_path = os.path.join(checkpoint_dir, folder)
    
    if os.path.isdir(folder_path) and folder.startswith("classifier_"):
        time_str = folder.split("_")[1]
        folder_dt = datetime.strptime(time_str, "%Y%m%d-%H%M%S")

        if folder_dt < threshold_dt:
            print(f"删除：{folder_path}")
            shutil.rmtree(folder_path)  # 删除整个文件夹


ValueError: time data 'stm' does not match format '%Y%m%d-%H%M%S'